In [ ]:
import os
import io
import sys
import math
import subprocess
import base64
from IPython.display import display, SVG

In [ ]:
class PrintAndDraw():
    def __init__(self, layout_engine="dot"):
        self.layout_engine = layout_engine
    def __enter__(self):
        self._stdout = sys.stdout
        sys.stdout = self._stringio = io.StringIO()
        return self
    def __exit__(self, *args):
        sys.stdout = self._stdout
        print("-----------")
        print(self._stringio.getvalue())
        self.run_gv()
        del self._stringio
    def run_gv(self):
        str_to_render = self._stringio.getvalue()
        cmd_l = [f"{self.layout_engine}", "-Tsvg"]
        rslt = subprocess.run(cmd_l,
                              input = str_to_render.encode(),
                              capture_output=True)
        if rslt.returncode:
            raise RuntimeError("An error occurred passing the input to"
                               f" {self.layout_engine}: {rslt.stderr.decode()}")
        else:
            display(SVG(data=rslt.stdout.decode()))

with PrintAndDraw():
    print("digraph {A->B->C->A}")

In [ ]:
class PrintAndSave():
    def __init__(self, fname, layout_engine="dot"):
        self.fname = fname
        self.layout_engine = layout_engine
    def __enter__(self):
        self._stdout = sys.stdout
        sys.stdout = self._stringio = io.StringIO()
        return self
    def __exit__(self, *args):
        sys.stdout = self._stdout
        print("-----------")
        print(self._stringio.getvalue())
        self.run_gv()
        del self._stringio
    def run_gv(self):
        str_to_render = self._stringio.getvalue()
        cmd_l = [f"{self.layout_engine}", "-Tsvg"]
        rslt = subprocess.run(cmd_l,
                              input = str_to_render.encode(),
                              capture_output=True)
        if rslt.returncode:
            raise RuntimeError("An error occurred passing the input to"
                               f" {self.layout_engine}: {rslt.stderr.decode()}")
        else:
            with open(self.fname, "w") as ofile:
                ofile.write(rslt.stdout.decode())

with PrintAndSave("/tmp/junk.svg"):
    print("digraph {A->B->C->A}")

In [ ]:
def get_label(longname):
    """
    Convert a long name to a short version
    """
    return "something"  # FIX ME

def get_rel_paths(here_path, root_path):
    """
    Given the current full path and the root path, return the
    relative path to the directory above, and the relative path
    to this full path
    """
    return "rel_path_to_parent", "rel_path_to_here"  # FIX ME

def remove_excluded_subdirs(dir_list):
    """
    For os.path.walk(), we can restrict the branches that get traversed by changing
    the list which walk() returns as 'subdirs'.  Remember that when we are editing
    that list, we are actually changing the memory representation inside the walk()
    generator, so the semantics are a little tricky.  For example, we can't simultaneously
    loop over the list and edit it.  I'll give you a working function for this task
    to avoid confusion.
    
    This function removes 'reveal.js', '.git', and anything starting with '_' from the list.
    """
    more_names_to_remove = [elt for elt in dir_list if elt.startswith('_')][:]
    for nm in ['reveal.js', '.git'] + more_names_to_remove:
        if nm in dir_list:
            dir_list.remove(nm)

def show_this_leaf(nm):
    """
    Use this function to return False for leaf node names you don't want to draw.
    """
    # FIX ME!
    return True

class Node():
    def __init__(self, parent_node, name):
        """
        parent_node should be the Node instance of the parent.
        
        name is a long name, like a full relative path.  It needs to be
        unique for all nodes in the tree.
        
        Note how we make a connection to the parent node when this node is created.
        """
        self.parent = parent_node
        if self.parent is not None:
            self.parent._add_kid(self)
        self.name = name
        self.label = get_label(self.name)
        self.kids = []
        self.descendant_count = 0
    def _add_kid(self, kid_node):
        self.kids.append(kid_node)
    def add_descendant_to_all_ancestors(self):
        """
        Modify this function so that when it is called for a node's parent,
        that parent and all ancestors get their descendant_count incremented.
        """
        pass  # FIX ME!
    def write_node(self, indent=0):
        print(f'{indent*" "}"{self.name}" [label="{self.label}"];') # FIX ME
    def traverse_node_defs(self, indent=0):
        """
        Write the DOT code that defines this Node and all its descendants.
        
        Here and in traverse_edge_defs(), 'indent' just helps with formatting
        when writing out the DOT code.
        """
        self.write_node(indent=indent)
        for kid in self.kids:
            kid.traverse_node_defs(indent+4)
    def write_incoming_edge(self, this_parent, indent=0):
        print(f'{indent*" "}"{this_parent.name}" -> "{self.name}" [];') # FIX ME
    def traverse_edge_defs(self, indent=0):
        """
        Write the DOT code that defines the incoming edges for this Node and
        all its descendants.
        
        Here and in traverse_node_defs(), 'indent' just helps with formatting
        when writing out the DOT code.
        """
        for kid in self.kids:
            kid.write_incoming_edge(self, indent=indent+4)
            kid.traverse_edge_defs(indent+4)


# Maintain a dictionary of Nodes so that we can find them by name.  Use
# paths relative to the root as keys- so the very first key is just '.'
nodes = {}
root_path = '/path/to/your/repo/docs'  # FIX ME
root_node = Node(None, '.')
nodes['.'] = root_node

# Some tests for common problems
assert os.path.isdir(root_path), "Something is wrong with the root_path value"
for test_pair, rslt_pair in [(("foo/bar/baz/blrfl", "foo/bar"), ("baz", "baz/blrfl")),
                             (("foo/bar/baz", "foo/bar"), (".", "baz")),
                             (("foo/bar", "foo/bar"), ("..", ".")),
                            ]:
    assert rslt_pair == get_rel_paths(*test_pair), f"get_rel_paths fails for {test_pair}"

# Walk the tree
for dirname, subdirs, files in os.walk(root_path):
    remove_excluded_subdirs(subdirs)
    rel_dir_path, rel_path = get_rel_paths(dirname, root_path)
    if rel_path in nodes:
        # This happens on the very first node
        dir_node = nodes[rel_path]
    else:
        assert rel_dir_path in nodes
        dir_node= Node(nodes[rel_dir_path], rel_path)
        nodes[rel_path] = dir_node

    # Add nodes for all the children of this dir
    for file in files:
        if show_this_leaf(file):
            full_path = os.path.join(dirname, file)
            ignore_this, rel_path = get_rel_paths(full_path, root_path)
            this_node = Node(dir_node, rel_path)
            nodes[rel_path] = this_node

# Calculate number of descendants for all nodes
for node in nodes:
    nodes[node].add_descendant_to_all_ancestors()

# Write out the Dot code
#with PrintAndSave("/tmp/junk.svg", layout_engine="sfdp"):
with PrintAndDraw(layout_engine="circo"):
    print("digraph {")
    root_node.traverse_node_defs()
    root_node.traverse_edge_defs()
    print("}")
